In [0]:
from pyspark.sql.functions import col, min, max, countDistinct, sum, when, datediff, first, lit
from pyspark.sql import Window

SILVER_SALES_TABLE = "workspace.retail.silver_sales"
GOLD_DIM_CUSTOMER = "workspace.retail.gold_dim_customer"

def build_customer_dimension(df_sales):
    max_date_row = df_sales.select(max(col("invoice_date_only")).alias("max_date")).first()
    dataset_today = max_date_row["max_date"]
    
    df_customers = df_sales.groupBy("customer_id").agg(
        first("country", ignorenulls=True).alias("country"),
        
        min("invoice_date_only").alias("first_purchase_date"),
        max("invoice_date_only").alias("last_purchase_date"),
        
        countDistinct("invoice").alias("total_orders_count"),
        
        sum(when(col("quantity") > 0, col("total_amount")).otherwise(0)).alias("total_spent"),
        sum(when(col("quantity") < 0, col("total_amount")).otherwise(0)).alias("total_returns_value")
    )
    
    df_rfm_base = df_customers.withColumn(
        "days_since_last_purchase", 
        datediff(lit(dataset_today), col("last_purchase_date"))
    )
    
    df_gold_customers = df_rfm_base.withColumn("rfm_segment",
        when((col("days_since_last_purchase") <= 30) & (col("total_orders_count") >= 5) & (col("total_spent") > 1000), "1. Champion")
        .when((col("days_since_last_purchase") <= 30), "2. Active/New")
        .when((col("days_since_last_purchase") > 90) & (col("total_orders_count") >= 5), "3. At Risk VIP")
        .when((col("days_since_last_purchase") > 90), "4. Churned (Lost)")
        .otherwise("5. Regular")
    )
    
    return df_gold_customers

def main():
    df_sales = spark.table(SILVER_SALES_TABLE)
    gold_customers_df = build_customer_dimension(df_sales)
    
    gold_customers_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(GOLD_DIM_CUSTOMER)
        
    print(f"Success! Created dimension for {gold_customers_df.count()} unique customers.")

if __name__ == "__main__":
    main()